# 07 — Persiapan Model Klasifikasi Citra Awan

Notebook ini menyiapkan arsitektur model deep learning untuk klasifikasi jenis awan
menggunakan PyTorch dan `timm`.

Tahap ini tidak melakukan training, validasi, maupun evaluasi menggunakan test set.

Tujuan notebook:

1. Memuat pemetaan kelas dari metadata tahap sebelumnya.
2. Membaca ukuran input citra dari konfigurasi preprocessing.
3. Menentukan perangkat komputasi CPU atau GPU.
4. Membentuk model ResNet18.
5. Menggunakan bobot pralatih ImageNet jika tersedia.
6. Mengganti classification head sesuai jumlah kelas dataset.
7. Membekukan backbone untuk tahap awal transfer learning.
8. Memeriksa jumlah parameter model.
9. Melakukan forward-pass menggunakan tensor sintetis.
10. Memeriksa fungsi loss tanpa memperbarui bobot.
11. Menyimpan konfigurasi model ke `dataset/processed/`.
12. Menyiapkan model untuk notebook training berikutnya.

Data test tidak dibaca atau digunakan pada notebook ini.

## Mengimpor Library

Library pada notebook ini hanya digunakan untuk membaca metadata, mengatur
reproduksibilitas, membangun model, dan menyimpan konfigurasi model.

Notebook tidak mengimpor ulang OpenCV, Albumentations, atau class Dataset karena
seluruh proses tersebut sudah ditempatkan pada notebook sebelumnya.

In [7]:
from pathlib import Path
from datetime import datetime
import json
import random
import sys
import warnings

try:
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    import timm

    from IPython.display import display

except ImportError as exc:
    raise ImportError(
        "Dependency persiapan model belum lengkap.\n\n"
        "Aktifkan .venv, kemudian jalankan melalui terminal VS Code:\n\n"
        r".\.venv\Scripts\python.exe -m pip install -r requirements.txt"
    ) from exc


pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)


print(f"Python      : {sys.version.split()[0]}")
print(f"NumPy       : {np.__version__}")
print(f"Pandas      : {pd.__version__}")
print(f"PyTorch     : {torch.__version__}")
print(f"timm        : {timm.__version__}")
print(f"CUDA        : {torch.cuda.is_available()}")

Python      : 3.11.9
NumPy       : 2.4.6
Pandas      : 3.0.5
PyTorch     : 2.13.0+cpu
timm        : 1.0.28
CUDA        : False


## Menentukan Lokasi Project

Lokasi project ditentukan secara dinamis sehingga notebook dapat dijalankan dari
folder utama project maupun dari folder `notebooks`.

Notebook tidak menggunakan path absolut Docker `/app`.

In [8]:
CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR


DATASET_DIR = PROJECT_DIR / "dataset"
PROCESSED_DIR = DATASET_DIR / "processed"
MODELS_DIR = PROJECT_DIR / "models"
LOGS_DIR = PROJECT_DIR / "logs"


CLASS_MAPPING_PATH = (
    PROCESSED_DIR
    / "class_mapping.json"
)

PREPROCESSING_CONFIG_PATH = (
    PROCESSED_DIR
    / "preprocessing_config.json"
)

DATALOADER_CONFIG_PATH = (
    PROCESSED_DIR
    / "dataloader_config.json"
)

MODEL_CONFIG_PATH = (
    PROCESSED_DIR
    / "model_config.json"
)

MODEL_PARAMETER_SUMMARY_PATH = (
    PROCESSED_DIR
    / "model_parameter_summary.csv"
)


path_table = pd.DataFrame({
    "Nama": [
        "PROJECT_DIR",
        "PROCESSED_DIR",
        "MODELS_DIR",
        "CLASS_MAPPING_PATH",
        "PREPROCESSING_CONFIG_PATH",
        "DATALOADER_CONFIG_PATH",
        "MODEL_CONFIG_PATH",
    ],
    "Path": [
        PROJECT_DIR,
        PROCESSED_DIR,
        MODELS_DIR,
        CLASS_MAPPING_PATH,
        PREPROCESSING_CONFIG_PATH,
        DATALOADER_CONFIG_PATH,
        MODEL_CONFIG_PATH,
    ],
})

path_table["Ada"] = path_table["Path"].map(Path.exists)

display(path_table)

,Nama,Path,Ada
0,PROJECT_DIR,C:\Users\jardm\Documents\n8n-logsiswaparalayang\cloud-classification,True
1,PROCESSED_DIR,C:\Users\jardm\Documents\n8n-logsiswaparalayang\cloud-classification\dataset\processed,True
2,MODELS_DIR,C:\Users\jardm\Documents\n8n-logsiswaparalayang\cloud-classification\models,True
3,CLASS_MAPPING_PATH,C:\Users\jardm\Documents\n8n-logsiswaparalayang\cloud-classification\dataset\processed\class_mapping.json,True
4,PREPROCESSING_CONFIG_PATH,C:\Users\jardm\Documents\n8n-logsiswaparalayang\cloud-classification\dataset\processed\preprocessing_config.json,True
5,DATALOADER_CONFIG_PATH,C:\Users\jardm\Documents\n8n-logsiswaparalayang\cloud-classification\dataset\processed\dataloader_config.json,True
6,MODEL_CONFIG_PATH,C:\Users\jardm\Documents\n8n-logsiswaparalayang\cloud-classification\dataset\processed\model_config.json,False


## Memeriksa Prasyarat

Persiapan model memerlukan pemetaan kelas dan konfigurasi preprocessing.

`dataloader_config.json` bersifat dianjurkan. Apabila file tersebut tersedia,
nilai seed akan digunakan kembali agar konsisten dengan notebook DataLoader.

In [9]:
required_directories = [
    PROJECT_DIR,
    DATASET_DIR,
    PROCESSED_DIR,
    MODELS_DIR,
    LOGS_DIR,
]

required_files = [
    CLASS_MAPPING_PATH,
    PREPROCESSING_CONFIG_PATH,
]


missing_directories = [
    path
    for path in required_directories
    if not path.is_dir()
]

missing_files = [
    path
    for path in required_files
    if not path.is_file()
]


if missing_directories:
    formatted_paths = "\n".join(
        f"- {path}"
        for path in missing_directories
    )

    raise FileNotFoundError(
        "Folder project berikut belum tersedia:\n"
        f"{formatted_paths}"
    )


if missing_files:
    formatted_paths = "\n".join(
        f"- {path}"
        for path in missing_files
    )

    raise FileNotFoundError(
        "Metadata tahap sebelumnya belum tersedia:\n"
        f"{formatted_paths}\n\n"
        "Selesaikan notebook 05 dan 06 sebelum melanjutkan."
    )


if not DATALOADER_CONFIG_PATH.is_file():
    warnings.warn(
        "dataloader_config.json belum ditemukan. "
        "Notebook akan menggunakan seed default 42.",
        stacklevel=2,
    )


print("Prasyarat utama persiapan model tersedia.")

Prasyarat utama persiapan model tersedia.


## Membaca Konfigurasi Sebelumnya

Notebook membaca konfigurasi yang telah disimpan tanpa menghitung ulang
dataset, transformasi, distribusi kelas, atau DataLoader.

In [10]:
with CLASS_MAPPING_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    raw_class_mapping = json.load(file)


with PREPROCESSING_CONFIG_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    preprocessing_config = json.load(file)


if DATALOADER_CONFIG_PATH.is_file():
    with DATALOADER_CONFIG_PATH.open(
        "r",
        encoding="utf-8",
    ) as file:
        dataloader_config = json.load(file)
else:
    dataloader_config = {}


print("Pemetaan kelas dan konfigurasi preprocessing berhasil dibaca.")

Pemetaan kelas dan konfigurasi preprocessing berhasil dibaca.


## Menormalisasi Pemetaan Kelas

Struktur `class_mapping.json` dapat berupa pemetaan langsung atau berada
di dalam kunci `class_to_idx`. Fungsi berikut membuat proses pembacaan
lebih stabil tanpa menuliskan jumlah maupun nama kelas secara manual.

In [11]:
def extract_class_to_idx(mapping: dict) -> dict[str, int]:
    """
    Mengubah beberapa bentuk class mapping menjadi format:
    {
        "nama_kelas": indeks
    }
    """
    if not isinstance(mapping, dict):
        raise TypeError(
            "Isi class_mapping.json harus berupa dictionary."
        )

    if "class_to_idx" in mapping:
        candidate = mapping["class_to_idx"]

    elif (
        mapping
        and all(isinstance(value, int) for value in mapping.values())
    ):
        candidate = mapping

    elif (
        mapping
        and all(str(key).isdigit() for key in mapping.keys())
        and all(isinstance(value, str) for value in mapping.values())
    ):
        candidate = {
            class_name: int(class_idx)
            for class_idx, class_name in mapping.items()
        }

    elif (
        "classes" in mapping
        and isinstance(mapping["classes"], list)
    ):
        candidate = {
            class_name: class_idx
            for class_idx, class_name in enumerate(mapping["classes"])
        }

    else:
        raise ValueError(
            "Struktur class_mapping.json tidak dikenali. "
            "Gunakan class_to_idx, daftar classes, atau mapping langsung."
        )

    class_to_idx = {
        str(class_name): int(class_idx)
        for class_name, class_idx in candidate.items()
    }

    class_to_idx = dict(
        sorted(
            class_to_idx.items(),
            key=lambda item: item[1],
        )
    )

    expected_indices = list(range(len(class_to_idx)))
    actual_indices = list(class_to_idx.values())

    if actual_indices != expected_indices:
        raise ValueError(
            "Indeks kelas harus berurutan mulai dari 0. "
            f"Indeks ditemukan: {actual_indices}"
        )

    return class_to_idx


class_to_idx = extract_class_to_idx(raw_class_mapping)

idx_to_class = {
    class_idx: class_name
    for class_name, class_idx in class_to_idx.items()
}

NUM_CLASSES = len(class_to_idx)

if NUM_CLASSES < 2:
    raise ValueError(
        "Klasifikasi memerlukan sedikitnya dua kelas."
    )


class_table = pd.DataFrame({
    "class_idx": list(idx_to_class.keys()),
    "class_name": list(idx_to_class.values()),
})

display(class_table)

print(f"Jumlah kelas: {NUM_CLASSES}")

,class_idx,class_name
0,0,1_cumulus
1,1,2_altocumulus
2,2,3_cirrus
3,3,4_clearsky
4,4,5_stratocumulus
5,5,6_cumulonimbus
6,6,7_mixed


Jumlah kelas: 7


## Mengambil Ukuran Input Model

Ukuran input dibaca dari `preprocessing_config.json`. Beberapa variasi
nama parameter didukung agar notebook tidak bergantung pada satu bentuk
konfigurasi saja.

Apabila ukuran tidak ditemukan, nilai awal 224 × 224 digunakan dan
peringatan akan ditampilkan.

In [12]:
def iterate_dicts(value):
    """
    Menghasilkan seluruh dictionary di dalam objek bertingkat.
    """
    if isinstance(value, dict):
        yield value

        for nested_value in value.values():
            yield from iterate_dicts(nested_value)

    elif isinstance(value, list):
        for nested_value in value:
            yield from iterate_dicts(nested_value)


def parse_size_value(value):
    """
    Mengubah nilai ukuran menjadi pasangan (height, width).
    """
    if isinstance(value, int):
        return value, value

    if isinstance(value, (list, tuple)) and len(value) >= 2:
        return int(value[0]), int(value[1])

    if isinstance(value, dict):
        height = (
            value.get("height")
            or value.get("target_height")
            or value.get("image_height")
        )

        width = (
            value.get("width")
            or value.get("target_width")
            or value.get("image_width")
        )

        if height is not None and width is not None:
            return int(height), int(width)

    return None


def extract_input_size(config: dict) -> tuple[int, int]:
    """
    Mencari ukuran citra dari konfigurasi preprocessing.
    """
    size_keys = [
        "image_size",
        "input_size",
        "target_size",
        "resize",
    ]

    for current_dict in iterate_dicts(config):
        for key in size_keys:
            if key in current_dict:
                parsed_size = parse_size_value(
                    current_dict[key]
                )

                if parsed_size is not None:
                    return parsed_size

        height = (
            current_dict.get("target_height")
            or current_dict.get("image_height")
        )

        width = (
            current_dict.get("target_width")
            or current_dict.get("image_width")
        )

        if height is not None and width is not None:
            return int(height), int(width)

    warnings.warn(
        "Ukuran input tidak ditemukan dalam preprocessing_config.json. "
        "Nilai fallback 224 × 224 digunakan.",
        stacklevel=2,
    )

    return 224, 224


IMAGE_HEIGHT, IMAGE_WIDTH = extract_input_size(
    preprocessing_config
)

INPUT_CHANNELS = 3


input_table = pd.DataFrame({
    "Parameter": [
        "INPUT_CHANNELS",
        "IMAGE_HEIGHT",
        "IMAGE_WIDTH",
        "NUM_CLASSES",
    ],
    "Nilai": [
        INPUT_CHANNELS,
        IMAGE_HEIGHT,
        IMAGE_WIDTH,
        NUM_CLASSES,
    ],
})

display(input_table)

,Parameter,Nilai
0,INPUT_CHANNELS,3
1,IMAGE_HEIGHT,224
2,IMAGE_WIDTH,224
3,NUM_CLASSES,7


## Memeriksa Normalisasi Input

Model pretrained ImageNet sebaiknya menggunakan normalisasi yang sesuai
dengan distribusi input saat pretraining.

Pemeriksaan ini hanya membaca nilai normalisasi dari konfigurasi. Pipeline
transform tidak dibuat ulang.

In [14]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def extract_normalization(config: dict):
    """
    Mencari pasangan mean dan standard deviation dari konfigurasi.
    """
    for current_dict in iterate_dicts(config):
        if "mean" in current_dict and "std" in current_dict:
            mean = current_dict["mean"]
            std = current_dict["std"]

            if (
                isinstance(mean, (list, tuple))
                and isinstance(std, (list, tuple))
                and len(mean) == 3
                and len(std) == 3
            ):
                return (
                    [float(value) for value in mean],
                    [float(value) for value in std],
                )

    return None, None


NORMALIZATION_MEAN, NORMALIZATION_STD = extract_normalization(
    preprocessing_config
)


if NORMALIZATION_MEAN is None:
    normalization_status = "Tidak ditemukan dalam konfigurasi"

else:
    mean_compatible = np.allclose(
        NORMALIZATION_MEAN,
        IMAGENET_MEAN,
        atol=1e-6,
    )

    std_compatible = np.allclose(
        NORMALIZATION_STD,
        IMAGENET_STD,
        atol=1e-6,
    )

    if mean_compatible and std_compatible:
        normalization_status = "Sesuai ImageNet"
    else:
        normalization_status = "Berbeda dari ImageNet"


normalization_table = pd.DataFrame({
    "Parameter": [
        "Mean preprocessing",
        "Std preprocessing",
        "Mean ImageNet",
        "Std ImageNet",
        "Status",
    ],
    "Nilai": [
        NORMALIZATION_MEAN,
        NORMALIZATION_STD,
        IMAGENET_MEAN,
        IMAGENET_STD,
        normalization_status,
    ],
})

display(normalization_table)


if normalization_status == "Berbeda dari ImageNet":
    warnings.warn(
        "Normalisasi preprocessing berbeda dari ImageNet. "
        "Periksa kembali konfigurasi notebook 05 sebelum training.",
        stacklevel=2,
    )

,Parameter,Nilai
0,Mean preprocessing,"[0.485, 0.456, 0.406]"
1,Std preprocessing,"[0.229, 0.224, 0.225]"
2,Mean ImageNet,"[0.485, 0.456, 0.406]"
3,Std ImageNet,"[0.229, 0.224, 0.225]"
4,Status,Sesuai ImageNet


## Menetapkan Reproduksibilitas dan Perangkat

Seed dari konfigurasi DataLoader digunakan kembali apabila tersedia.
Urutan pemilihan perangkat adalah CUDA, Apple MPS, kemudian CPU.

In [15]:
SEED = int(
    dataloader_config.get(
        "seed",
        dataloader_config.get("SEED", 42),
    )
)


def seed_everything(seed: int) -> None:
    """
    Menetapkan seed untuk Python, NumPy, dan PyTorch.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


seed_everything(SEED)


if torch.cuda.is_available():
    DEVICE = torch.device("cuda")

elif (
    hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    DEVICE = torch.device("mps")

else:
    DEVICE = torch.device("cpu")


device_table = pd.DataFrame({
    "Parameter": [
        "Seed",
        "Device",
        "CUDA tersedia",
        "Jumlah GPU CUDA",
    ],
    "Nilai": [
        SEED,
        str(DEVICE),
        torch.cuda.is_available(),
        torch.cuda.device_count(),
    ],
})

display(device_table)

,Parameter,Nilai
0,Seed,42
1,Device,cpu
2,CUDA tersedia,False
3,Jumlah GPU CUDA,0


## Menentukan Konfigurasi Model

ResNet18 digunakan sebagai baseline transfer learning. Model ini cukup
ringan untuk tahap awal eksperimen dan classifier dapat disesuaikan secara
otomatis menggunakan jumlah kelas dataset.

Pada fase awal, backbone dibekukan dan hanya classifier yang dapat diperbarui.
Backbone dapat dibuka kembali pada tahap fine-tuning di notebook training.

In [16]:
MODEL_NAME = "resnet18"

PRETRAINED_REQUESTED = True

# Jika bobot pretrained tidak dapat diunduh, notebook tetap dapat
# melakukan pemeriksaan arsitektur menggunakan inisialisasi acak.
ALLOW_RANDOM_INIT_FALLBACK = True

FREEZE_BACKBONE = True

DROPOUT_RATE = 0.20

LABEL_SMOOTHING = 0.10


available_models = timm.list_models()

if MODEL_NAME not in available_models:
    raise ValueError(
        f"Model '{MODEL_NAME}' tidak ditemukan pada timm {timm.__version__}."
    )


model_configuration_table = pd.DataFrame({
    "Parameter": [
        "MODEL_NAME",
        "PRETRAINED_REQUESTED",
        "FREEZE_BACKBONE",
        "DROPOUT_RATE",
        "LABEL_SMOOTHING",
        "NUM_CLASSES",
    ],
    "Nilai": [
        MODEL_NAME,
        PRETRAINED_REQUESTED,
        FREEZE_BACKBONE,
        DROPOUT_RATE,
        LABEL_SMOOTHING,
        NUM_CLASSES,
    ],
})

display(model_configuration_table)

,Parameter,Nilai
0,MODEL_NAME,resnet18
1,PRETRAINED_REQUESTED,True
2,FREEZE_BACKBONE,True
3,DROPOUT_RATE,0.2
4,LABEL_SMOOTHING,0.1
5,NUM_CLASSES,7


## Membuat Fungsi Model

`timm.create_model()` membangun backbone dan mengganti classifier akhir
berdasarkan nilai `num_classes`.

Apabila bobot pretrained gagal diunduh, fallback menggunakan bobot acak
hanya dilakukan jika `ALLOW_RANDOM_INIT_FALLBACK=True`. Kondisi sebenarnya
tetap dicatat pada konfigurasi akhir.

In [17]:
def create_cloud_classifier(
    model_name: str,
    num_classes: int,
    pretrained: bool,
    dropout_rate: float,
    allow_random_fallback: bool,
):
    """
    Membuat model klasifikasi menggunakan timm.
    """
    try:
        model = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=num_classes,
            in_chans=INPUT_CHANNELS,
            drop_rate=dropout_rate,
        )

        pretrained_loaded = pretrained
        initialization_message = (
            "Bobot pretrained berhasil dimuat."
            if pretrained
            else "Model menggunakan inisialisasi acak."
        )

    except Exception as exc:
        if not pretrained or not allow_random_fallback:
            raise RuntimeError(
                "Model gagal dibuat. Jika kendala berasal dari unduhan "
                "bobot pretrained, periksa koneksi internet atau ubah "
                "PRETRAINED_REQUESTED menjadi False."
            ) from exc

        warnings.warn(
            "Bobot pretrained tidak dapat dimuat. "
            "Model dibuat menggunakan inisialisasi acak. "
            f"Pesan awal: {exc}",
            stacklevel=2,
        )

        model = timm.create_model(
            model_name,
            pretrained=False,
            num_classes=num_classes,
            in_chans=INPUT_CHANNELS,
            drop_rate=dropout_rate,
        )

        pretrained_loaded = False
        initialization_message = (
            "Fallback aktif: model menggunakan inisialisasi acak."
        )

    return model, pretrained_loaded, initialization_message


def configure_trainable_parameters(
    model: nn.Module,
    freeze_backbone: bool,
) -> nn.Module:
    """
    Mengatur parameter yang dapat dilatih.

    freeze_backbone=True:
        hanya classifier yang trainable.

    freeze_backbone=False:
        seluruh parameter model trainable.
    """
    if freeze_backbone:
        model.requires_grad_(False)

        classifier = model.get_classifier()
        classifier.requires_grad_(True)

    else:
        model.requires_grad_(True)

    return model

In [18]:
model, PRETRAINED_LOADED, initialization_message = (
    create_cloud_classifier(
        model_name=MODEL_NAME,
        num_classes=NUM_CLASSES,
        pretrained=PRETRAINED_REQUESTED,
        dropout_rate=DROPOUT_RATE,
        allow_random_fallback=ALLOW_RANDOM_INIT_FALLBACK,
    )
)


model = configure_trainable_parameters(
    model=model,
    freeze_backbone=FREEZE_BACKBONE,
)


model = model.to(DEVICE)


classifier_module = model.get_classifier()

classifier_names = [
    module_name
    for module_name, module in model.named_modules()
    if module is classifier_module
]

CLASSIFIER_NAME = (
    classifier_names[0]
    if classifier_names
    else classifier_module.__class__.__name__
)


print(initialization_message)
print(f"Model             : {MODEL_NAME}")
print(f"Classifier        : {CLASSIFIER_NAME}")
print(f"Jumlah kelas      : {NUM_CLASSES}")
print(f"Perangkat         : {DEVICE}")
print(f"Pretrained aktual : {PRETRAINED_LOADED}")
print()
print(classifier_module)

model.safetensors: reconstructing file:   0%|          |  0.00B / 46.8MB            

model.safetensors: downloading bytes:           |  0.00B            

Bobot pretrained berhasil dimuat.
Model             : resnet18
Classifier        : fc
Jumlah kelas      : 7
Perangkat         : cpu
Pretrained aktual : True

Linear(in_features=512, out_features=7, bias=True)


## Menghitung Parameter Model

Parameter dibedakan menjadi parameter total, trainable, dan frozen.
Pada konfigurasi awal, hanya classifier yang seharusnya trainable.

In [19]:
parameter_records = []

for parameter_name, parameter in model.named_parameters():
    top_level_module = parameter_name.split(".")[0]

    parameter_records.append({
        "module": top_level_module,
        "parameter_name": parameter_name,
        "shape": str(tuple(parameter.shape)),
        "num_parameters": parameter.numel(),
        "trainable": parameter.requires_grad,
    })


parameter_df = pd.DataFrame(parameter_records)


parameter_summary = (
    parameter_df
    .groupby("module", as_index=False)
    .agg(
        total_parameters=("num_parameters", "sum"),
        trainable_parameters=(
            "num_parameters",
            lambda values: int(
                parameter_df.loc[
                    values.index,
                    "num_parameters",
                ][
                    parameter_df.loc[
                        values.index,
                        "trainable",
                    ]
                ].sum()
            ),
        ),
    )
)


parameter_summary["frozen_parameters"] = (
    parameter_summary["total_parameters"]
    - parameter_summary["trainable_parameters"]
)


TOTAL_PARAMETERS = int(
    parameter_df["num_parameters"].sum()
)

TRAINABLE_PARAMETERS = int(
    parameter_df.loc[
        parameter_df["trainable"],
        "num_parameters",
    ].sum()
)

FROZEN_PARAMETERS = (
    TOTAL_PARAMETERS
    - TRAINABLE_PARAMETERS
)

TRAINABLE_PERCENTAGE = (
    100.0
    * TRAINABLE_PARAMETERS
    / TOTAL_PARAMETERS
)


display(parameter_summary)


overall_parameter_table = pd.DataFrame({
    "Jenis": [
        "Total parameter",
        "Trainable parameter",
        "Frozen parameter",
        "Persentase trainable",
    ],
    "Jumlah": [
        TOTAL_PARAMETERS,
        TRAINABLE_PARAMETERS,
        FROZEN_PARAMETERS,
        f"{TRAINABLE_PERCENTAGE:.4f}%",
    ],
})

display(overall_parameter_table)

,module,total_parameters,trainable_parameters,frozen_parameters
0,bn1,128,0,128
1,conv1,9408,0,9408
2,fc,3591,3591,0
3,layer1,147968,0,147968
4,layer2,525568,0,525568
5,layer3,2099712,0,2099712
6,layer4,8393728,0,8393728


,Jenis,Jumlah
0,Total parameter,11180103
1,Trainable parameter,3591
2,Frozen parameter,11176512
3,Persentase trainable,0.0321%


In [20]:
trainable_parameter_table = (
    parameter_df
    .loc[
        parameter_df["trainable"],
        [
            "parameter_name",
            "shape",
            "num_parameters",
        ],
    ]
    .reset_index(drop=True)
)


if trainable_parameter_table.empty:
    raise RuntimeError(
        "Tidak terdapat parameter trainable pada model."
    )


if FREEZE_BACKBONE:
    unexpected_trainable = trainable_parameter_table[
        ~trainable_parameter_table["parameter_name"].str.startswith(
            f"{CLASSIFIER_NAME}."
        )
    ]

    if not unexpected_trainable.empty:
        warnings.warn(
            "Terdapat parameter trainable di luar classifier. "
            "Periksa kembali struktur model.",
            stacklevel=2,
        )


display(trainable_parameter_table)

print(
    f"Jumlah tensor parameter trainable: "
    f"{len(trainable_parameter_table)}"
)

,parameter_name,shape,num_parameters
0,fc.weight,"(7, 512)",3584
1,fc.bias,"(7,)",7


Jumlah tensor parameter trainable: 2


## Memeriksa Forward Pass

Pemeriksaan dilakukan menggunakan tensor sintetis. Dengan demikian,
notebook tidak perlu membuat ulang DataLoader dan tidak mengakses data test.

Output model harus mempunyai bentuk:

`[ukuran_batch, jumlah_kelas]`

In [21]:
DUMMY_BATCH_SIZE = 2


dummy_images = torch.randn(
    DUMMY_BATCH_SIZE,
    INPUT_CHANNELS,
    IMAGE_HEIGHT,
    IMAGE_WIDTH,
    device=DEVICE,
)


model.eval()

with torch.inference_mode():
    dummy_logits = model(dummy_images)


expected_output_shape = (
    DUMMY_BATCH_SIZE,
    NUM_CLASSES,
)


if tuple(dummy_logits.shape) != expected_output_shape:
    raise RuntimeError(
        "Dimensi output model tidak sesuai.\n"
        f"Diharapkan : {expected_output_shape}\n"
        f"Ditemukan  : {tuple(dummy_logits.shape)}"
    )


if not torch.isfinite(dummy_logits).all():
    raise RuntimeError(
        "Output model mengandung NaN atau infinity."
    )


forward_table = pd.DataFrame({
    "Komponen": [
        "Input",
        "Output logits",
        "Output yang diharapkan",
        "Tipe data output",
    ],
    "Nilai": [
        str(tuple(dummy_images.shape)),
        str(tuple(dummy_logits.shape)),
        str(expected_output_shape),
        str(dummy_logits.dtype),
    ],
})

display(forward_table)

print("Forward pass berhasil.")

,Komponen,Nilai
0,Input,"(2, 3, 224, 224)"
1,Output logits,"(2, 7)"
2,Output yang diharapkan,"(2, 7)"
3,Tipe data output,torch.float32


Forward pass berhasil.


## Memeriksa Probabilitas dan Fungsi Loss

`CrossEntropyLoss` menerima logits secara langsung. Oleh karena itu,
Softmax tidak ditempatkan di dalam classifier model.

Softmax hanya digunakan pada bagian ini untuk memeriksa bahwa probabilitas
setiap sampel berjumlah mendekati satu.

In [22]:
with torch.inference_mode():
    dummy_probabilities = torch.softmax(
        dummy_logits,
        dim=1,
    )


probability_sums = dummy_probabilities.sum(dim=1)


if not torch.allclose(
    probability_sums,
    torch.ones_like(probability_sums),
    atol=1e-6,
):
    raise RuntimeError(
        "Jumlah probabilitas setiap sampel tidak sama dengan satu."
    )


criterion = nn.CrossEntropyLoss(
    label_smoothing=LABEL_SMOOTHING,
)


dummy_targets = (
    torch.arange(DUMMY_BATCH_SIZE, device=DEVICE)
    % NUM_CLASSES
).long()


dummy_loss = criterion(
    dummy_logits,
    dummy_targets,
)


if not torch.isfinite(dummy_loss):
    raise RuntimeError(
        "Nilai loss sintetis tidak finite."
    )


probability_table = pd.DataFrame({
    "Sampel": list(range(DUMMY_BATCH_SIZE)),
    "Jumlah probabilitas": (
        probability_sums
        .detach()
        .cpu()
        .numpy()
    ),
    "Target sintetis": (
        dummy_targets
        .detach()
        .cpu()
        .numpy()
    ),
})

display(probability_table)

print(f"Loss sintetis: {dummy_loss.item():.6f}")
print("Pemeriksaan fungsi loss berhasil.")

,Sampel,Jumlah probabilitas,Target sintetis
0,0,1.0,0
1,1,1.0,1


Loss sintetis: 1.674230
Pemeriksaan fungsi loss berhasil.


## Status Model Setelah Pemeriksaan

Notebook tidak melakukan optimisasi maupun pembaruan bobot. Model dikembalikan
ke mode evaluasi setelah pemeriksaan forward pass.

Pada notebook training, mode training harus diaktifkan kembali menggunakan
`model.train()`.

In [23]:
model.eval()

print(f"Mode training model: {model.training}")
print("Tidak ada pembaruan bobot yang dilakukan.")

Mode training model: False
Tidak ada pembaruan bobot yang dilakukan.


## Menyimpan Konfigurasi Model

Notebook hanya menyimpan metadata dan ringkasan parameter. Bobot model awal
tidak disimpan karena folder `models/` dikhususkan untuk checkpoint hasil
training.

Informasi `pretrained_loaded` mencatat kondisi aktual, termasuk ketika
fallback ke inisialisasi acak terjadi.

In [24]:
model_config = {
    "created_at": datetime.now().astimezone().isoformat(
        timespec="seconds"
    ),
    "model_name": MODEL_NAME,
    "framework": "PyTorch",
    "torch_version": torch.__version__,
    "timm_version": timm.__version__,
    "seed": SEED,
    "device_used_during_validation": str(DEVICE),
    "input_channels": INPUT_CHANNELS,
    "image_height": IMAGE_HEIGHT,
    "image_width": IMAGE_WIDTH,
    "input_shape": [
        INPUT_CHANNELS,
        IMAGE_HEIGHT,
        IMAGE_WIDTH,
    ],
    "num_classes": NUM_CLASSES,
    "class_to_idx": class_to_idx,
    "idx_to_class": {
        str(class_idx): class_name
        for class_idx, class_name in idx_to_class.items()
    },
    "pretrained_requested": PRETRAINED_REQUESTED,
    "pretrained_loaded": PRETRAINED_LOADED,
    "allow_random_init_fallback": ALLOW_RANDOM_INIT_FALLBACK,
    "freeze_backbone_initial": FREEZE_BACKBONE,
    "classifier_name": CLASSIFIER_NAME,
    "dropout_rate": DROPOUT_RATE,
    "normalization": {
        "mean": NORMALIZATION_MEAN,
        "std": NORMALIZATION_STD,
        "compatibility_status": normalization_status,
    },
    "loss": {
        "name": "CrossEntropyLoss",
        "label_smoothing": LABEL_SMOOTHING,
        "input": "logits",
    },
    "parameters": {
        "total": TOTAL_PARAMETERS,
        "trainable": TRAINABLE_PARAMETERS,
        "frozen": FROZEN_PARAMETERS,
        "trainable_percentage": TRAINABLE_PERCENTAGE,
    },
    "forward_validation": {
        "dummy_batch_size": DUMMY_BATCH_SIZE,
        "output_shape": list(dummy_logits.shape),
        "finite_output": True,
        "dummy_loss": float(dummy_loss.item()),
    },
    "source_artifacts": {
        "class_mapping": CLASS_MAPPING_PATH.name,
        "preprocessing_config": PREPROCESSING_CONFIG_PATH.name,
        "dataloader_config": (
            DATALOADER_CONFIG_PATH.name
            if DATALOADER_CONFIG_PATH.is_file()
            else None
        ),
    },
    "test_data_used": False,
}


with MODEL_CONFIG_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        model_config,
        file,
        indent=4,
        ensure_ascii=False,
    )


parameter_summary.to_csv(
    MODEL_PARAMETER_SUMMARY_PATH,
    index=False,
    encoding="utf-8",
)


print(f"Konfigurasi model : {MODEL_CONFIG_PATH}")
print(f"Ringkasan parameter: {MODEL_PARAMETER_SUMMARY_PATH}")

Konfigurasi model : C:\Users\jardm\Documents\n8n-logsiswaparalayang\cloud-classification\dataset\processed\model_config.json
Ringkasan parameter: C:\Users\jardm\Documents\n8n-logsiswaparalayang\cloud-classification\dataset\processed\model_parameter_summary.csv


In [25]:
with MODEL_CONFIG_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    saved_model_config = json.load(file)


required_model_config_keys = {
    "model_name",
    "input_shape",
    "num_classes",
    "class_to_idx",
    "pretrained_loaded",
    "freeze_backbone_initial",
    "loss",
    "parameters",
    "forward_validation",
}


missing_config_keys = (
    required_model_config_keys
    - set(saved_model_config.keys())
)


if missing_config_keys:
    raise KeyError(
        "Metadata model belum lengkap. Kunci yang hilang: "
        f"{sorted(missing_config_keys)}"
    )


if saved_model_config["num_classes"] != NUM_CLASSES:
    raise ValueError(
        "Jumlah kelas pada metadata model tidak konsisten."
    )


if saved_model_config["input_shape"] != [
    INPUT_CHANNELS,
    IMAGE_HEIGHT,
    IMAGE_WIDTH,
]:
    raise ValueError(
        "Dimensi input pada metadata model tidak konsisten."
    )


if not MODEL_PARAMETER_SUMMARY_PATH.is_file():
    raise FileNotFoundError(
        "Ringkasan parameter model gagal disimpan."
    )


print("Metadata model berhasil disimpan dan divalidasi.")

Metadata model berhasil disimpan dan divalidasi.


In [26]:
final_status = pd.DataFrame({
    "Pemeriksaan": [
        "Pemetaan kelas",
        "Ukuran input",
        "Arsitektur model",
        "Classifier",
        "Forward pass",
        "CrossEntropyLoss",
        "Metadata model",
        "Ringkasan parameter",
        "Data test digunakan",
    ],
    "Status": [
        "Berhasil",
        f"{IMAGE_HEIGHT} × {IMAGE_WIDTH}",
        MODEL_NAME,
        CLASSIFIER_NAME,
        "Berhasil",
        "Berhasil",
        "Tersimpan",
        "Tersimpan",
        "Tidak",
    ],
})


display(final_status)


assert tuple(dummy_logits.shape) == (
    DUMMY_BATCH_SIZE,
    NUM_CLASSES,
)

assert torch.isfinite(dummy_logits).all()
assert torch.isfinite(dummy_loss)
assert MODEL_CONFIG_PATH.is_file()
assert MODEL_PARAMETER_SUMMARY_PATH.is_file()


print("=" * 70)
print("PERSIAPAN MODEL SELESAI")
print("=" * 70)
print(f"Model             : {MODEL_NAME}")
print(f"Jumlah kelas      : {NUM_CLASSES}")
print(f"Input             : {INPUT_CHANNELS} × {IMAGE_HEIGHT} × {IMAGE_WIDTH}")
print(f"Pretrained aktual : {PRETRAINED_LOADED}")
print(f"Backbone dibekukan: {FREEZE_BACKBONE}")
print(f"Total parameter   : {TOTAL_PARAMETERS:,}")
print(f"Trainable          : {TRAINABLE_PARAMETERS:,}")
print(f"Device             : {DEVICE}")
print("=" * 70)

,Pemeriksaan,Status
0,Pemetaan kelas,Berhasil
1,Ukuran input,224 × 224
2,Arsitektur model,resnet18
3,Classifier,fc
4,Forward pass,Berhasil
5,CrossEntropyLoss,Berhasil
6,Metadata model,Tersimpan
7,Ringkasan parameter,Tersimpan
8,Data test digunakan,Tidak


PERSIAPAN MODEL SELESAI
Model             : resnet18
Jumlah kelas      : 7
Input             : 3 × 224 × 224
Pretrained aktual : True
Backbone dibekukan: True
Total parameter   : 11,180,103
Trainable          : 3,591
Device             : cpu


## Kesimpulan

Tahap persiapan model klasifikasi jenis awan telah selesai.

Hasil utama:

1. Jumlah output model ditentukan secara otomatis dari `class_mapping.json`.
2. Ukuran input mengikuti konfigurasi preprocessing.
3. ResNet18 berhasil dibangun menggunakan library `timm`.
4. Backbone dibekukan untuk fase awal transfer learning.
5. Classifier akhir tetap menjadi parameter trainable.
6. Forward pass menghasilkan logits dengan dimensi yang sesuai.
7. `CrossEntropyLoss` berhasil menerima logits dan label sintetis.
8. Tidak ada pembaruan bobot model pada notebook ini.
9. Data test tidak digunakan.
10. Konfigurasi model dan ringkasan parameter telah disimpan.

File yang dihasilkan:

- `dataset/processed/model_config.json`
- `dataset/processed/model_parameter_summary.csv`

Tahap berikutnya adalah membuat `08_training_model.ipynb` untuk:

- membaca konfigurasi model;
- membangun kembali Dataset dan DataLoader secara modular;
- menjalankan fase training classifier;
- melakukan fine-tuning backbone;
- mencatat train loss dan validation loss;
- mencatat train accuracy dan validation accuracy;
- menerapkan early stopping;
- menyimpan checkpoint terbaik berdasarkan performa validation;
- tidak menggunakan test set selama pemilihan model.